# Ejercicio 5 — Comparación $3^3$ vs. Box-Behnken (BBD)

**Objetivo.** Ajustar el modelo RSM de segundo orden sobre el $3^3$ completo (27 corridas)
y sobre un diseño Box-Behnken equivalente (15 corridas) con los mismos factores. Comparar:
(a) coeficientes estimados, (b) errores estándar, (c) calidad de la superficie resultante.

**Factores:**
- $A$ = Temperatura de extracción: 60 (−1), 75 (0), 90 °C (+1)
- $B$ = Tiempo de extracción: 30 (−1), 45 (0), 60 min (+1)
- $C$ = Concentración de solvente: 2 (−1), 3 (0), 4 % (+1)

**Respuesta:** Pureza del fármaco (%)

**Dataset $3^3$:** `../../datos/pureza-farmaco-3k.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import linalg

df3k = pd.read_csv('../../datos/pureza-farmaco-3k.csv')
print(f'3^3: {len(df3k)} corridas'); print(df3k.head())

## 1. Construir el Box-Behnken (BBD) con 3 factores

El BBD para $k=3$ tiene $12 + n_c$ corridas. La estructura estándar combina
pares de factores en sus extremos (±1) mientras el tercero está en 0,
más puntos centrales.

In [ ]:
np.random.seed(7)

# Estructura BBD para k=3: 12 corridas de borde + 3 centros
bbd_bordes = pd.DataFrame({
    'x1': [-1,-1, 1, 1, -1,-1, 1, 1,  0, 0, 0, 0],
    'x2': [-1, 1,-1, 1,  0, 0, 0, 0, -1,-1, 1, 1],
    'x3': [ 0, 0, 0, 0, -1, 1,-1, 1, -1, 1,-1, 1]
})
bbd_centros = pd.DataFrame({'x1':[0,0,0], 'x2':[0,0,0], 'x3':[0,0,0]})
bbd = pd.concat([bbd_bordes, bbd_centros], ignore_index=True)

# Ajustar modelo 3^3 para generar respuestas del BBD.
# ⚠ NOTA PEDAGÓGICA: las respuestas del BBD se simulan prediciendo con el modelo del 3^3
# y añadiendo ruido pequeño. Esto hace que ambos modelos sean similares por construcción.
# En un experimento real, los datos del BBD serían observaciones independientes y los
# modelos ajustados diferirían más. El objetivo aquí es comparar la ESTRUCTURA del diseño
# (número de corridas, GL para error) más que los resultados empíricos.
formula3k = 'pureza ~ x1+x2+x3+I(x1**2)+I(x2**2)+I(x3**2)+x1:x2+x1:x3+x2:x3'
modelo3k = smf.ols(formula3k, data=df3k).fit()
bbd['pureza'] = modelo3k.predict(bbd) + np.random.normal(0, 0.5, len(bbd))

print(f'BBD: {len(bbd)} corridas (12 borde + 3 centros)')
print(bbd)

## 2. Ajustar modelo de 2° orden en ambos diseños

In [ ]:
formula = 'pureza ~ x1+x2+x3+I(x1**2)+I(x2**2)+I(x3**2)+x1:x2+x1:x3+x2:x3'
modelo_bbd = smf.ols(formula, data=bbd).fit()

print('=== Modelo 3^3 (27 corridas) ===')
print(modelo3k.summary().tables[1])
print('\n=== Modelo BBD (15 corridas) ===')
print(modelo_bbd.summary().tables[1])

## 3. Tabla comparativa de coeficientes y errores estándar

In [ ]:
terminos = ['Intercept','x1','x2','x3',
            'I(x1 ** 2)','I(x2 ** 2)','I(x3 ** 2)',
            'x1:x2','x1:x3','x2:x3']
nombres  = ['β₀','β₁','β₂','β₃','β₁₁','β₂₂','β₃₃','β₁₂','β₁₃','β₂₃']

comp = pd.DataFrame({
    'Término': nombres,
    '3^3 est.': [modelo3k.params[t] for t in terminos],
    'BBD est.': [modelo_bbd.params[t] for t in terminos],
    '3^3 SE':   [modelo3k.bse[t] for t in terminos],
    'BBD SE':   [modelo_bbd.bse[t] for t in terminos],
})
comp['ΔRSE (%)'] = ((comp['BBD SE'] - comp['3^3 SE']) / comp['3^3 SE'] * 100).round(1)
print(comp.round(4).to_string(index=False))

## 4. Comparación del punto óptimo

In [ ]:
def punto_estacionario(modelo):
    p = modelo.params
    B = np.array([
        [p['I(x1 ** 2)'],  p['x1:x2']/2,   p['x1:x3']/2],
        [p['x1:x2']/2,   p['I(x2 ** 2)'],   p['x2:x3']/2],
        [p['x1:x3']/2,   p['x2:x3']/2,   p['I(x3 ** 2)']]
    ])
    b = np.array([p['x1'], p['x2'], p['x3']]) / 2
    xs = -linalg.solve(B, b)
    return xs

xs_3k  = punto_estacionario(modelo3k)
xs_bbd = punto_estacionario(modelo_bbd)

centros_r = np.array([75, 45, 3])
deltas_r  = np.array([15, 15, 1])

print('Punto estacionario (codificado):')
print(f'  3^3: x1={xs_3k[0]:.3f}, x2={xs_3k[1]:.3f}, x3={xs_3k[2]:.3f}')
print(f'  BBD: x1={xs_bbd[0]:.3f}, x2={xs_bbd[1]:.3f}, x3={xs_bbd[2]:.3f}')

real_3k  = centros_r + xs_3k * deltas_r
real_bbd = centros_r + xs_bbd * deltas_r
print('\nPunto estacionario (real):')
print(f'  3^3: T={real_3k[0]:.1f}°C, t={real_3k[1]:.1f}min, C={real_3k[2]:.2f}%')
print(f'  BBD: T={real_bbd[0]:.1f}°C, t={real_bbd[1]:.1f}min, C={real_bbd[2]:.2f}%')

## 5. Comparación visual de superficies (par A×B, C en óptimo)

In [ ]:
x1g, x2g = np.meshgrid(np.linspace(-1.4, 1.4, 50), np.linspace(-1.4, 1.4, 50))

def sup(m, x1g, x2g, x3_val):
    p = m.params
    return (p['Intercept'] + p['x1']*x1g + p['x2']*x2g + p['x3']*x3_val
            + p['I(x1 ** 2)']*x1g**2 + p['I(x2 ** 2)']*x2g**2 + p['I(x3 ** 2)']*x3_val**2
            + p['x1:x2']*x1g*x2g + p['x1:x3']*x1g*x3_val + p['x2:x3']*x2g*x3_val)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
vmin, vmax = 80, 96
for ax, mod, xs, titulo, n in zip(
    axes, [modelo3k, modelo_bbd], [xs_3k, xs_bbd],
    ['$3^3$ (27 corridas)', 'Box-Behnken (15 corridas)'], [27, 15]
):
    zg = sup(mod, x1g, x2g, xs[2])
    cp = ax.contourf(x1g, x2g, zg, levels=12, cmap='RdYlGn', vmin=vmin, vmax=vmax)
    plt.colorbar(cp, ax=ax, label='Pureza (%)')
    ax.plot(xs[0], xs[1], 'r*', markersize=14, label='Óptimo')
    ax.set_xlabel('$x_1$ (Temperatura)', fontsize=11)
    ax.set_ylabel('$x_2$ (Tiempo)', fontsize=11)
    ax.set_title(f'{titulo}\n(C fijado en x3={xs[2]:.2f})')
    ax.legend()

plt.suptitle('Comparación de superficies RSM: $3^3$ vs. Box-Behnken', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Tabla resumen de eficiencia

In [ ]:
resumen = pd.DataFrame({
    'Criterio': ['Corridas totales', 'Parámetros en 2° orden (k=3)',
                 'GL para el error', 'Rotatibilidad',
                 'Niveles por factor', 'Puntos fuera del cubo unitario'],
    '$3^3$':       ['27',  '10', '17', 'No', '3 (−1,0,+1)', 'No'],
    'Box-Behnken': ['15',  '10',  '5', 'Aprox.', '3 (−1,0,+1)', 'No']
})
print(resumen.to_string(index=False))

## 7. Conclusión

- El **$3^3$** y el **Box-Behnken** ajustan el mismo modelo de segundo orden (10 parámetros).
- El $3^3$ usa 27 corridas vs. 15 del BBD: **80% más corridas** (o el BBD usa 44% menos) sin ganancia sustancial en precisión de los coeficientes para este tamaño de problema.
- El BBD es preferido industrialmente para $k=3$ o $4$: no tiene puntos de esquina (donde varias variables están simultáneamente en sus extremos), lo que reduce riesgos operativos.
- El $3^k$ se justifica cuando se quiere estimar **todos los términos de interacción de alto orden** (incluidos los cuadráticos cruzados $A_Q B_Q$, etc.), que el BBD no estima.
- **Limitación de este ejemplo:** las superficies son similares porque los datos del BBD se simularon a partir del modelo $3^3$. En datos reales independientes habría más divergencia entre los dos ajustes.

> **Regla práctica.** Usa $3^k$ cuando necesitas la estructura completa de contrastes ortogonales (estudio de componentes L/Q). Usa BBD o CCD cuando el objetivo es la optimización RSM con el mínimo de corridas.